 # Creating your own benchmark and mechanism



 This is a **VS Code / Jupyter percent-format notebook**.



 Save as:



 `custom_benchmark_mechanism_tutorial.py`



 Then use:



 **Jupyter: Export Current Python File as Jupyter Notebook**



 ---



 This tutorial begins with the mathematical problem definition and only then

 maps that model to the code abstraction.



 The goal is to create a new benchmark that can reuse the same mechanism

 algorithms and bilevel optimization infrastructure.

 # 1. Start from the mathematical model



 Before creating classes, define the multi-agent controlled dynamical system:



 \[

 \mathcal E

 =

 (

 \mathcal S,

 \{\mathcal A_i\}_{i=1}^{N},

 P,

 \{R_i\}_{i=1}^{N},

 \{\mathcal O_i\}_{i=1}^{N},

 \gamma

 ).

 \]



 You need to specify:



 1. state \(s_t\in\mathcal S\);

 2. each agent action \(a_{i,t}\);

 3. transition \(P(s_{t+1}\mid s_t,a_t)\);

 4. intrinsic/base reward \(R_i\);

 5. observation function \(O_i\);

 6. initial-state/reset distribution;

 7. horizon/termination semantics.



 Only after the benchmark is mathematically clear should a mechanism be

 introduced.

 # 2. Separate benchmark dynamics from regulation



 The benchmark answers:



 > What happens in the world when delivered actions are executed?



 The mechanism answers:



 > What actions reach the world, what reward reaches the learner, and what

 > information reaches the learner?



 Mechanism intervention:



 \[

 o_t^\*

 =

 \mathcal M_\theta^O(s_t,o_t),

 \]



 \[

 a_t^\*

 =

 \mathcal M_\theta^A(s_t,a_t),

 \]



 \[

 r_t^\*

 =

 \mathcal M_\theta^R(

 r_t,

 s_t,

 a_t^\*,

 s_{t+1}

 ).

 \]



 The transition must consume the regulated action:



 \[

 s_{t+1}

 \sim

 P(\cdot\mid s_t,a_t^\*).

 \]



 Keeping this boundary clean is what makes one quota algorithm reusable in a

 fishery, irrigation system, energy market, traffic network, or another

 benchmark.

 # 3. Ingredients of a custom benchmark



 A benchmark needs:



 ```text

 1. state representation

 2. agent IDs / multi-agent mappings

 3. action semantics

 4. reset hook

 5. transition hook

 6. intrinsic/base reward hook

 7. observation hook

 8. action and observation spaces

 9. runtime context / info values

 10. mechanism bindings

 ```



 A custom mechanism needs:



 ```text

 1. semantic parameters

 2. dimension

 3. encode()

 4. decode()

 5. clip()

 6. param_names()

 7. to_vector()

 8. optional action()

 9. optional reward()

 10. optional observation()

 11. runtime bindings when benchmark context is required

 ```

 # 4. Hooks



 The environment hooks mark benchmark-specific functions.



 Conceptually:



 ```python

 @reset

 def initialize_state(...):

     ...



 @transition

 def dynamics(...):

     ...



 @observation

 def make_observation(...):

     ...



 @reward

 def intrinsic_reward(...):

     ...



 @action

 def benchmark_action_preprocessing(...):

     ...

 ```



 `MultiAgentRegulatedEnv.__init_subclass__` discovers the decorated functions.



 That lets the generic environment own the lifecycle while a benchmark owns

 only domain-specific equations.

 # 5. Schemas and runtime context



 It is useful to distinguish two meanings of "schema".



 ## 5.1 Runtime/world context schema



 Each step can publish structured information such as:



 ```text

 env_id

 seed

 policy_seed

 train/eval status

 mechanism_id

 observation

 reward

 action

 info

 ```



 Domain-specific diagnostics belong in `info`, for example:



 ```text

 resource stock

 biological growth

 realized harvest

 restoration

 quota violation

 reservoir level

 ```



 This gives the optimizer/world layer a traceable description of what happened.



 ## 5.2 Metrics/reporting schema



 If the typed metrics/visualization branch is merged, important benchmark

 quantities should also be represented in a `MetricSchema`.



 The runtime context answers:



 > What happened on this step?



 The metrics schema answers:



 > How should this quantity accumulate, reduce, and be queried for reporting?



 Keep those responsibilities separate.

 # 6. Worked benchmark: renewable common-pool resource



 We create a minimal renewable resource.



 State:



 \[

 x_t\in[0,K].

 \]



 Normalized state:



 \[

 \bar x_t=\frac{x_t}{K}.

 \]



 Each agent chooses one extraction fraction:



 \[

 a_{i,t}

 =

 \begin{bmatrix}

 u_{i,t}

 \end{bmatrix},

 \qquad

 u_{i,t}\in[0,1].

 \]



 Logistic growth:



 \[

 G(x_t)

 =

 r x_t

 \left(

 1-\frac{x_t}{K}

 \right).

 \]



 Maximum extraction capacity for agent \(i\):



 \[

 H_i^{\max}.

 \]



 Attempted extraction:



 \[

 H_{i,t}=u_{i,t}H_i^{\max}.

 \]



 Transition:



 \[

 x_{t+1}

 =

 \operatorname{clip}

 \left(

 x_t+G(x_t)-\sum_iH_{i,t},

 0,

 K

 \right).

 \]



 Base reward:



 \[

 r_{i,t}=H_{i,t}.

 \]



 Observation:



 \[

 o_{i,t}=[\bar x_t].

 \]

In [ ]:
from __future__ import annotations

from typing import Any

import numpy as np

# Target imports:
#
# from core.envs.marl_regulated import MultiAgentRegulatedEnv
# from core.envs.hooks import action, observation, reset, reward, transition
# from core.types import MultiAgentDict


 # 7. Implement the benchmark class



 The following cells show the intended current abstraction.



 Keep the constructor benchmark-specific. Do not encode quota/subsidy logic

 directly inside it.

In [ ]:
# class RenewableResourceEnv(MultiAgentRegulatedEnv):
#     def __init__(
#         self,
#         *,
#         ecology_cfg: dict[str, Any],
#         **kwargs,
#     ):
#         super().__init__(**kwargs)
#
#         self.r = float(
#             ecology_cfg.get("r", 0.3)
#         )
#
#         self.K = float(
#             ecology_cfg.get("K", 1_000.0)
#         )
#
#         self.x_init = float(
#             ecology_cfg.get(
#                 "x_init",
#                 0.8 * self.K,
#             )
#         )
#
#         self.max_harvest_per_agent = float(
#             ecology_cfg.get(
#                 "max_harvest_per_agent",
#                 0.02 * self.K,
#             )
#         )


 ## 7.1 Reset hook



 Mathematically:



 \[

 x_0\sim p_0(x).

 \]



 Begin with a deterministic reset so the model can be validated exactly.

In [ ]:
#     @reset
#     def reset_resource(
#         self,
#     ) -> dict[str, float]:
#         return {
#             "resource": self.x_init,
#             "last_usage": 0.0,
#         }


 ## 7.2 Transition hook



 The transition receives the **already regulated action** \(a_t^\*\).



 That is the core mechanism-design contract.

In [ ]:
#     @transition
#     def renewable_transition(
#         self,
#         *,
#         A_t: MultiAgentDict,
#         S_t: dict[str, float],
#         **kwargs,
#     ) -> dict[str, float]:
#         x = float(
#             S_t["resource"]
#         )
#
#         requested_harvest = {
#             agent_id: (
#                 float(
#                     np.asarray(action)
#                     .reshape(-1)[0]
#                 )
#                 * self.max_harvest_per_agent
#             )
#             for agent_id, action
#             in A_t.items()
#         }
#
#         H = float(
#             sum(
#                 requested_harvest.values()
#             )
#         )
#
#         growth = (
#             self.r
#             * x
#             * (1.0 - x / self.K)
#         )
#
#         available = max(
#             x + growth,
#             0.0,
#         )
#
#         realized_harvest = min(
#             H,
#             available,
#         )
#
#         x_next = float(
#             np.clip(
#                 available
#                 - realized_harvest,
#                 0.0,
#                 self.K,
#             )
#         )
#
#         self._update_infos(
#             key="resource",
#             values=x,
#         )
#
#         self._update_infos(
#             key="resource_norm",
#             values=x / self.K,
#         )
#
#         self._update_infos(
#             key="growth",
#             values=growth,
#         )
#
#         self._update_infos(
#             key="realized_harvest",
#             values=realized_harvest,
#         )
#
#         return {
#             "resource": x_next,
#             "last_usage": realized_harvest,
#         }


 ## 7.3 Base reward hook



 The benchmark defines the intrinsic reward **before reward shaping**.



 Here:



 \[

 r_{i,t}=H_{i,t}.

 \]



 The exact hook signature should match the final merged base-env lifecycle.

In [ ]:
#     @reward
#     def harvest_reward(
#         self,
#         action_dict: MultiAgentDict,
#     ) -> MultiAgentDict:
#         return {
#             agent_id: float(
#                 np.asarray(action)
#                 .reshape(-1)[0]
#                 * self.max_harvest_per_agent
#             )
#             for agent_id, action
#             in action_dict.items()
#         }


 ## 7.4 Observation hook



 The benchmark observation is intentionally minimal:



 \[

 o_{i,t}=[x_t/K].

 \]



 Mechanisms can augment it later.

In [ ]:
#     @observation
#     def resource_observation(
#         self,
#         observation_dict: MultiAgentDict,
#     ) -> MultiAgentDict:
#         resource_norm = (
#             self.S_t["resource"]
#             / self.K
#         )
#
#         obs = np.array(
#             [resource_norm],
#             dtype=np.float32,
#         )
#
#         return {
#             agent_id: obs.copy()
#             for agent_id
#             in self.agents
#         }


 # 8. Define action semantics explicitly



 Document every action component.



 For the example:



 ```text

 component 0 -> extraction fraction

 ```



 If the benchmark later becomes:



 ```text

 component 0 -> extraction fraction

 component 1 -> restoration effort

 component 2 -> investment

 ```



 then mechanism `action_component` values must reference this map.



 Never make a mechanism infer action meaning from array position without a

 documented contract.

 # 9. Add an existing quota mechanism



 The generic quota asks only for a normalized resource level:



 \[

 b_t=x_t/K.

 \]

In [ ]:
# from core.mechanism.algorithms.quota import QuotaMechanism
#
# quota = QuotaMechanism(
#     fixed_quota=0.50,
#     action_component=0,
#     bindings={
#         "resource_level": lambda env: (
#             env.S_t["resource"] / env.K
#         ),
#     },
# )


 This is the abstraction boundary:



 ```text

 Quota:

     "I require a normalized resource_level."



 Benchmark:

     "For me, resource_level = resource / K."

 ```



 The quota therefore does not need to know whether the resource is fish,

 groundwater, a reservoir, or another stock.

 # 10. Create a custom mechanism from mathematics



 We create a scarcity-weighted extraction tax.

 ## 10.1 Mathematical definition



 Let:



 \[

 \bar x_t=x_t/K

 \]



 and:



 \[

 \lambda\in[0,1].

 \]



 For extraction fraction \(u_{i,t}\):



 \[

 r_{i,t}^{*}

 =

 r_{i,t}

 -

 \lambda

 (1-\bar x_t)

 u_{i,t}.

 \]



 Properties:



 - high resource -> small tax;

 - low resource -> larger tax;

 - zero extraction -> zero tax;

 - mechanism does not change physical dynamics.

 ## 10.2 Implement the custom mechanism



 A custom mechanism should satisfy the complete `Mechanism` vector/optimizer

 contract.

In [ ]:
# from dataclasses import dataclass, field, replace
# from typing import Callable
#
# from core.mechanism.base import Mechanism
#
#
# @dataclass(frozen=True)
# class ScarcityTaxMechanism(Mechanism):
#     tax_rate: float
#     action_component: int = 0
#
#     bindings: dict[
#         str,
#         Callable[[Any], Any],
#     ] = field(
#         default_factory=dict,
#         repr=False,
#         compare=False,
#     )
#
#     def __post_init__(
#         self,
#     ) -> None:
#         if not 0.0 <= self.tax_rate <= 1.0:
#             raise ValueError(
#                 "tax_rate must be in [0, 1]."
#             )
#
#         if "resource_level" not in self.bindings:
#             raise ValueError(
#                 "ScarcityTaxMechanism requires "
#                 "'resource_level' binding."
#             )
#
#     @property
#     def dimension(
#         self,
#     ) -> int:
#         return 1
#
#     def encode(
#         self,
#     ) -> np.ndarray:
#         return np.array(
#             [self.tax_rate],
#             dtype=np.float32,
#         )
#
#     def decode(
#         self,
#         x: np.ndarray,
#     ) -> "ScarcityTaxMechanism":
#         x = self._validate(x)
#
#         return replace(
#             self,
#             tax_rate=float(x[0]),
#         ).clip()
#
#     def clip(
#         self,
#     ) -> "ScarcityTaxMechanism":
#         return replace(
#             self,
#             tax_rate=float(
#                 np.clip(
#                     self.tax_rate,
#                     0.0,
#                     1.0,
#                 )
#             ),
#         )
#
#     def param_names(
#         self,
#     ) -> list[str]:
#         return [
#             "tax_rate",
#         ]
#
#     def to_vector(
#         self,
#     ) -> np.ndarray:
#         return np.array(
#             [self.tax_rate],
#             dtype=np.float32,
#         )
#
#     def reward(
#         self,
#         reward_dict: MultiAgentDict,
#         **kwargs,
#     ) -> MultiAgentDict:
#         resource_level = float(
#             kwargs["resource_level"]
#         )
#
#         actions = kwargs[
#             "action_after"
#         ]
#
#         scarcity = (
#             1.0 - resource_level
#         )
#
#         return {
#             agent_id: float(
#                 reward
#                 -
#                 self.tax_rate
#                 * scarcity
#                 * float(
#                     actions[agent_id][
#                         self.action_component
#                     ]
#                 )
#             )
#             for agent_id, reward
#             in reward_dict.items()
#         }


 # 11. Why each mechanism method exists



 `dimension`



 : Number of optimizer-controlled scalar dimensions.



 `encode()`



 : Mechanism -> normalized optimizer vector.



 `decode(x)`



 : Optimizer vector -> immutable mechanism instance.



 `clip()`



 : Enforce valid semantic bounds.



 `param_names()`



 : Names that correspond exactly to `encode()`.



 `to_vector()`



 : Semantic mechanism representation that may be exposed to agents.



 `encode()` and `to_vector()` can happen to match for a simple mechanism, but

 they serve different purposes.

 # 12. Fixed versus optimized mechanisms



 A fixed mechanism can expose:



 \[

 d=0.

 \]



 Then:



 ```python

 dimension == 0

 encode() -> np.empty(0)

 decode(np.empty(0)) -> self

 ```



 An optimized mechanism has:



 \[

 d>0.

 \]



 Composition adds dimensions:



 \[

 d_{\text{composition}}

 =

 \sum_i d_i.

 \]

 # 13. Compose the quota and custom tax



 Suppose:



 ```python

 mechanism = ChainedMechanism(

     children=(

         quota,

         tax,

     )

 )

 ```



 Action path:



 ```text

 policy action

   -> quota

   -> tax(identity action)

   -> transition

 ```



 Reward path:



 ```text

 base reward

   -> quota(identity reward)

   -> scarcity tax

   -> learner

 ```



 Because these mechanisms alter different channels, order has little effect.

 If two mechanisms both transform reward or action, chain order can change the

 result.

 # 14. Bindings are dependency injection



 Avoid benchmark-specific code inside a generic mechanism:



 ```python

 env.S_t["fish"]

 ```



 Prefer a semantic dependency:



 ```python

 bindings={

     "resource_level": lambda env: ...

 }

 ```



 A mechanism can request:



 ```text

 resource_level

 previous_actions

 agent_ids

 demand

 capacity

 price

 network_load

 ...

 ```



 The benchmark decides how those values are computed.

 # 15. Test the benchmark equation before RL



 Do not start with APPO or ES.



 First test the transition as a pure mathematical function.

In [ ]:
def logistic_step(
    x: float,
    harvest: float,
    *,
    r: float,
    K: float,
) -> float:
    growth = (
        r
        * x
        * (1.0 - x / K)
    )

    return float(
        np.clip(
            x
            + growth
            - harvest,
            0.0,
            K,
        )
    )


In [ ]:
x_next = logistic_step(
    800.0,
    50.0,
    r=0.3,
    K=1_000.0,
)

x_next


 Manual verification:



 \[

 G(800)

 =

 0.3(800)(1-0.8)

 =

 48.

 \]



 Therefore:



 \[

 x_{t+1}

 =

 800+48-50

 =

 798.

 \]

In [ ]:
assert np.isclose(
    x_next,
    798.0,
)


 # 16. Test the custom mechanism analytically



 Choose:



 ```text

 tax_rate       = 0.5

 resource_level = 0.2

 scarcity       = 0.8

 action         = 0.4

 base reward    = 1.0

 ```



 Expected:



 \[

 r^\*

 =

 1.0

 -

 0.5(0.8)(0.4)

 =

 0.84.

 \]

In [ ]:
expected_reward = (
    1.0
    -
    0.5
    * (1.0 - 0.2)
    * 0.4
)

assert np.isclose(
    expected_reward,
    0.84,
)


 Every mechanism should have an analytical test like this before it is used in

 a distributed RL run.

 # 17. Test encode/decode round trips



 For every optimized mechanism:



 \[

 M

 \xrightarrow{\text{encode}}

 x

 \xrightarrow{\text{decode}}

 M'

 \]



 should preserve semantic parameters up to floating-point tolerance.



 Example:



 ```python

 x = mechanism.encode()

 reconstructed = mechanism.decode(x)



 np.testing.assert_allclose(

     reconstructed.encode(),

     x,

 )



 assert (

     reconstructed.param_names()

     ==

     mechanism.param_names()

 )

 ```

 # 18. Test one complete regulated step



 For a deterministic fixture:



 1. set the environment state manually;

 2. supply a known raw policy action;

 3. compute expected normalized action;

 4. compute expected mechanism action transform;

 5. compute expected transition;

 6. compute expected base reward;

 7. compute expected mechanism reward;

 8. compute expected transformed observation;

 9. compare each intermediate value.



 This is much easier to debug than checking only final episode return.

 # 19. Action normalization



 The current regulated environment maps raw policy outputs

 \(z\in\mathbb R\) through:



 \[

 a=\sigma(z/T)

 \]



 with:



 \[

 T=4.

 \]



 Mechanisms therefore receive normalized semantic action components in

 \([0,1]\).



 If your benchmark needs another physical range, add an explicit conversion.

 Do not make mechanisms guess whether the input is a logit, normalized

 fraction, or physical unit.

 # 20. Observation spaces must include mechanism augmentation



 If:



 \[

 o_t\in\mathbb R^{d_o}

 \]



 and a mechanism appends \(d_m\) features:



 \[

 o_t^\*\in\mathbb R^{d_o+d_m}.

 \]



 The declared Gymnasium space must match.



 Social peer-action observation adds:



 \[

 d_m=(N-1)d_a.

 \]



 Add a test such as:



 ```python

 obs, _ = env.reset()



 for agent_id, value in obs.items():

     assert env.observation_spaces[

         agent_id

     ].contains(value)

 ```

 # 21. Context publication



 A regulated step should publish the values actually used by the system:



 ```text

 env_id

 seed

 policy_seed

 status

 mechanism_id

 transformed observation

 regulated action

 shaped reward

 benchmark info

 ```



 If the published context disagrees with the values that drove the transition

 or learner, debugging and reproducibility become unreliable.

 # 22. Reproducibility checklist



 Record:



 ```text

 benchmark constants

 mechanism parameters

 action-component semantics

 env seed

 policy seed

 evaluation seeds

 horizon

 number of agents

 stochastic-noise model

 optimizer seed

 ```



 Validate in this order:



 1. deterministic transition;

 2. deterministic mechanism transform;

 3. deterministic full environment step;

 4. repeated seeded episode;

 5. inner RL;

 6. outer optimization.

 # 23. Minimal test suite for a new benchmark



 ```text

 test_reset_state

 test_transition_matches_equation

 test_transition_boundaries

 test_base_reward

 test_observation_shape

 test_action_semantics

 test_info_values

 test_context_publication



 test_mechanism_required_bindings

 test_mechanism_transform

 test_mechanism_encode_decode

 test_mechanism_clip

 test_mechanism_param_names



 test_benchmark_plus_mechanism_one_step

 test_benchmark_plus_mechanism_episode

 ```



 Only after these pass should the benchmark be used in an expensive bilevel

 run.

 # 24. Pull-request checklist for a new benchmark



 ## Theory



 - state is mathematically defined;

 - action semantics are mathematically defined;

 - transition equation is documented;

 - base reward is documented;

 - observation is documented;

 - mechanism intervention point is documented.



 ## Code



 - reset hook;

 - transition hook;

 - reward hook;

 - observation hook;

 - action/observation spaces;

 - info/context fields;

 - mechanism bindings;

 - optimizer-vector semantics for learned mechanism parameters.



 ## Tests



 - equations tested numerically;

 - mechanism tested independently;

 - composition tested;

 - deterministic end-to-end step tested;

 - seeded reproducibility tested.



 This order keeps the code tied to a scientific model rather than letting the

 class hierarchy define the experiment.